In [ ]:
import os
from elasticsearch import Elasticsearch

In [ ]:
username = os.getenv("ELASTICSEARCH_USERNAME", "elastic")
password = os.getenv("ELASTICSEARCH_PASSWORD")
es = Elasticsearch(
    os.getenv("ELASTICSEARCH_URL", "https://localhost:9200"),
    basic_auth=(username, password) if password else None,
    ca_certs=os.getenv("ELASTICSEARCH_CA_CERTS"),
)
es.ping()

## Prepare the data

In [ ]:
import pandas as pd

df = pd.read_csv("myntra_products_catalog.csv").loc[:499]
df.head()

In [ ]:
df.isna().value_counts()

In [ ]:
df.fillna("None", inplace=True)

## Convert the relevant field to Vector using BERT model

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-mpnet-base-v2')

In [ ]:
df["DescriptionVector"] = df["Description"].apply(lambda x: model.encode(x))

In [ ]:
df.head()

In [ ]:
es.ping()

## Create new index in ElasticSearch!

In [ ]:
from indexMapping import indexMapping

es.indices.create(index="all_products", mappings=indexMapping)

## Ingest the data into index

In [ ]:
record_list = df.to_dict("records")

In [ ]:
for record in record_list:
    try:
        es.index(index="all_products", document=record, id=record["ProductID"])
    except Exception as e:
        print(e)

In [ ]:
es.count(index="all_products")

## Search the data

In [ ]:
input_keyword = "Blue Shoes"
vector_of_input_keyword = model.encode(input_keyword)

query = {
    "field" : "DescriptionVector",
    "query_vector" : vector_of_input_keyword,
    "k" : 2,
    "num_candidates" : 500, 
}

res = es.knn_search(index="all_products", knn=query , source=["ProductName","Description"])
res["hits"]["hits"]